In [ ]:
# Date: August 5, 2025
# Program: open and merge player and team data files
# Output:
#   1. nhl_output/skater_team_data.csv
#   2. nhl_output/goalie_team_data.csv
# Author: Julian di Giovanni

%reset -f

import pandas as pd                 # for importing and working with data
import os                           # for directory definitions
from IPython.display import display # for displaying data in a Jupyter notebook
import gc                           # for selective clearing

#  Get current working directory
current_dir = os.getcwd()

# Create data, output and figures directories based on current_dir
data_dir = os.path.join(os.path.dirname(current_dir), 'hockeyanalytics/nhl_data_combined')
output_dir = os.path.join(os.path.dirname(current_dir), 'hockeyanalytics/nhl_output')
# figure_dir = os.path.join(os.path.dirname(current_dir), 'nhl_figures')

#### Import data ####

# Import skater data
data_path = os.path.join(data_dir,"nhl_skater_stats.csv")
df_skater = pd.read_csv(data_path)
print(df_skater.head())

# Import goalie data
data_path = os.path.join(data_dir,"nhl_goalie_stats.csv")
df_goalie = pd.read_csv(data_path)
print(df_goalie.head())

# Import teams data
data_path = os.path.join(data_dir,"nhl_team_stats.csv")
df_team = pd.read_csv(data_path)
print(df_team.head())

# Import teams code data
data_path = os.path.join(data_dir,"nhl_teams.csv")
df_tcode = pd.read_csv(data_path)
print(df_tcode.head())

### Match skater and teams data ###

## Split team appreviations for multiple teams played in a year

# Split the comma-separated values and expand into separate columns
teamplayed_cols = df_skater['team_abbrev'].str.split(',', expand=True)

# Add these as new columns to your dataframe
for i in range(teamplayed_cols.shape[1]):
    df_skater[f'team_{i+1}'] = teamplayed_cols[i].str.strip()  # strip() removes whitespace
del teamplayed_cols, i

## Match team code with team abbreviation in tcode dataset 
df_team = df_team.merge(df_tcode[['team_id', 'team_abbrev']], on='team_id', how='left')
df_team = df_team.drop_duplicates(subset=['team_abbrev', 'season']) # Eliminate duplication in seasons

## Merge function for mulitple skater-team
def merge_team_data(df_skater, df_team, team_col_name):
    # Automatically discover team columns that match the pattern 'team_#'
    team_columns = [col for col in df_skater.columns if col.startswith('team_') and col[5:].isdigit()]
    
    # Sort them to ensure proper order (team_1, team_2, team_10, etc.)
    team_columns.sort(key=lambda x: int(x.split('_')[1]))
    
    print(f"Found team columns: {team_columns}")
    
    df_result = df_skater.copy()
    
    for i, team_col in enumerate(team_columns, 1):
        print(f"Merging {team_col}...")
        df_result = df_result.merge(
            df_team,
            left_on=[team_col, 'season'],
            right_on=[team_col_name, 'season'],
            how='left',
            suffixes=('', f'_team{i}')
        )
        
        # Drop the redundant column if it's different from the original
        if team_col_name in df_result.columns and team_col_name != team_col:
            df_result = df_result.drop(team_col_name, axis=1)
    
    return df_result

# Usage - just specify the team column name in df_team
df_skater_team = merge_team_data(df_skater, df_team, 'team_abbrev')  # or whatever the actual name is

### Match goalie and teams data ###

## Split team appreviations for multiple teams played in a year

# Split the comma-separated values and expand into separate columns
teamplayed_cols = df_goalie['team_abbrev'].str.split(',', expand=True)

# Add these as new columns to your dataframe
for i in range(teamplayed_cols.shape[1]):
    df_goalie[f'team_{i+1}'] = teamplayed_cols[i].str.strip()  # strip() removes whitespace

# Add these as new columns to your dataframe
for i in range(teamplayed_cols.shape[1]):
    df_goalie[f'team_{i+1}'] = teamplayed_cols[i].str.strip()  # strip() removes whitespace
del teamplayed_cols, i

# Merge with team using previously written function for skater
df_goalie_team = merge_team_data(df_goalie, df_team, 'team_abbrev')

### Save merged data to new files ###

# Save each DataFrame to the output directory
df_skater_team.to_csv(os.path.join(output_dir, 'skater_team_data.csv'), index=False)
df_goalie_team.to_csv(os.path.join(output_dir, 'goalie_team_data.csv'), index=False)

# Load them back later
df_skater_team = pd.read_csv(os.path.join(output_dir, 'skater_team_data.csv'))
df_goalie_team = pd.read_csv(os.path.join(output_dir, 'goalie_team_data.csv'))


     season  player_id      player_name team_abbrev position  games_played  \
0  20052006    8466320  Niklas Nordgren     CAR,PIT        L            58   
1  20052006    8457421        Ian Moran         BOS        D            12   
2  20052006    8470064  Steven Goertzen         CBJ        R            39   
3  20052006    8470159      Boyd Gordon         WSH        C            25   
4  20052006    8460588    Eric Nickulas         BOS        R            16   

   goals  assists  points  plus_minus  ...  shooting_pct  \
0      4        2       6          -8  ...       0.10526   
1      1        1       2           0  ...       0.14285   
2      0        0       0         -17  ...       0.00000   
3      0        1       1          -4  ...       0.00000   
4      2        4       6           2  ...       0.14285   

   time_on_ice_per_game  face_off_win_pct  pp_goals  pp_points  sh_goals  \
0              424.8965           0.20000         0          0         0   
1              864

/var/folders/_k/2x462ht50b1blz8h4vxrq3600000gn/T/ipykernel_26700/2885225512.py:118: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df_skater_team = pd.read_csv(os.path.join(output_dir, 'skater_team_data.csv'))
